In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/iitmd_ba_2501164_FA-02"


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    f"{PROJECT_PATH}/cleaned_lending_club.csv"
)


## Hypothesis Test 1: Income and Default

**Null Hypothesis (H₀):**  
The mean annual income of defaulters is equal to the mean annual income of non-defaulters.

**Alternative Hypothesis (H₁):**  
The mean annual income of defaulters is not equal to the mean annual income of non-defaulters.

**Significance Level:** α = 0.05


In [ ]:
from scipy.stats import ttest_ind

income_defaulters = df[df['default'] == 1]['annual_inc']
income_non_defaulters = df[df['default'] == 0]['annual_inc']

t_stat, p_value = ttest_ind(
    income_defaulters,
    income_non_defaulters,
    equal_var=False
)

income_defaulters.mean(), income_non_defaulters.mean(), p_value


(np.float64(55975.87405440839),
 np.float64(59137.707444039566),
 np.float64(6.479667824720691e-14))

### Decision and Interpretation

The p-value obtained from the two-sample t-test is **6.48 × 10⁻¹⁴**, which is significantly lower than the significance level of 0.05.

Therefore, the null hypothesis is rejected. This indicates a statistically significant difference between the mean annual income of defaulters and non-defaulters. On average, borrowers who default have lower annual incomes compared to borrowers who successfully repay their loans.


### Business Implication

The results suggest that borrower income is an important determinant of repayment ability. Incorporating income-related features into a credit risk assessment framework can help lenders better identify high-risk applicants and reduce potential loan defaults without unnecessarily restricting credit access.


In [ ]:
# Clean employment length
df['emp_length_clean'] = df['emp_length'].str.replace('+', '', regex=False)
df['emp_length_clean'] = df['emp_length_clean'].str.replace(' years', '', regex=False)
df['emp_length_clean'] = df['emp_length_clean'].str.replace(' year', '', regex=False)
df['emp_length_clean'] = df['emp_length_clean'].replace('< 1', '0')
df['emp_length_clean'] = df['emp_length_clean'].astype(float)

# Create categories
df['emp_length_group'] = pd.cut(
    df['emp_length_clean'],
    bins=[-1, 2, 5, 10, 20],
    labels=['Low', 'Medium', 'High', 'Very High']
)


In [ ]:
contingency_table = pd.crosstab(
    df['emp_length_group'],
    df['default']
)

contingency_table


default,0,1
emp_length_group,,
Low,2854,748
Medium,2464,590
High,6208,1713


In [ ]:
from scipy.stats import chi2_contingency

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

chi2_stat, p_value


(np.float64(7.167064651990511), np.float64(0.027777405904179635))

### Decision and Interpretation

The p-value obtained from the chi-square test is **0.0278**, which is less than the significance level of 0.05.

Therefore, the null hypothesis is rejected. This indicates a statistically significant association between employment length and loan default. Borrowers with different levels of employment stability exhibit different default behaviors.


### Business Implication

The results suggest that employment stability plays a role in credit risk assessment. Borrowers with shorter employment histories tend to exhibit higher default risk compared to those with longer employment histories. Including employment length as a categorical risk factor can improve the effectiveness of credit scoring and loan approval decisions.


Although the relationship is statistically significant, the strength of association appears moderate, indicating that employment length should be used in combination with other risk factors rather than as a standalone decision variable.


## Hypothesis Test 3: Loan Grade and Default

**Null Hypothesis (H₀):**  
Loan grade and loan default are independent of each other.

**Alternative Hypothesis (H₁):**  
Loan grade and loan default are not independent of each other.

**Significance Level:** α = 0.05


In [ ]:
grade_default_table = pd.crosstab(
    df['grade'],
    df['default']
)

grade_default_table


default,0,1
grade,,
A,2146,145
B,3707,586
C,3244,924
D,1529,744
E,656,431
F,200,171
G,44,50


In [ ]:
from scipy.stats import chi2_contingency

chi2_stat, p_value, dof, expected = chi2_contingency(grade_default_table)

chi2_stat, p_value


(np.float64(1058.9973411376689), np.float64(1.5487786176072638e-225))

### Decision and Interpretation

The p-value obtained from the chi-square test is **1.55 × 10⁻²²⁵**, which is significantly lower than the significance level of 0.05.

Therefore, the null hypothesis is rejected. This indicates a very strong statistical association between loan grade and loan default. As loan grades deteriorate from A to G, default risk increases substantially, confirming loan grade as a key determinant of credit risk.


### Business Implication

The results confirm that loan grade is one of the most powerful indicators of borrower risk. Lower-grade loans require stricter underwriting standards and higher risk-based pricing, while higher-grade loans can be approved with greater confidence. Loan grade should therefore remain a central feature in credit scoring and lending decision frameworks.
